# Aggregation and Groupby

## Introduction

SQL aggregate functions collapse a set of rows into a single summary value — the total, the maximum, the average. Combined with `GROUP BY`, they let you compute those summaries for each distinct category in a column. `HAVING` then lets you filter on the aggregated result, something `WHERE` cannot do.

## Objectives

You will be able to:

- Use `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX` to summarize columns
- Apply `GROUP BY` to split results by category
- Create readable column labels with `AS` (aliasing)
- Filter grouped results with `HAVING`
- Combine `WHERE` (row-level filter) and `HAVING` (group-level filter) in a single query

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('data/grouping_data_with_sql/data.sqlite')
cur = conn.cursor()

Database schema for reference:

![CRM schema](assets/grouping_data_with_sql/Database-Schema.png)

---

## Aggregate Functions

| Function | Returns |
|----------|---------|
| `COUNT(col)` | Number of non-null values in `col` |
| `COUNT(*)` | Total rows in the result set |
| `SUM(col)` | Sum of all values |
| `AVG(col)` | Mean of all values |
| `MIN(col)` | Smallest value |
| `MAX(col)` | Largest value |

These functions operate on the whole result set unless combined with `GROUP BY`.

In [ ]:
# Total number of customers
cur.execute("SELECT COUNT(*) FROM customers;").fetchone()

In [ ]:
# Multi-stat summary for the payments table
cur.execute("""
    SELECT COUNT(*) AS num_payments,
           SUM(amount) AS total_revenue,
           AVG(amount) AS avg_payment,
           MIN(amount) AS min_payment,
           MAX(amount) AS max_payment
    FROM payments;
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

---

## GROUP BY

`GROUP BY` splits the table into one group per unique value in the specified column(s), then applies the aggregate function to each group independently.

```sql
SELECT city, COUNT(employeeNumber) AS num_employees
FROM offices
JOIN employees USING(officeCode)
GROUP BY city
ORDER BY num_employees DESC;
```

In [ ]:
# Employees per office city
cur.execute("""
    SELECT city, COUNT(employeeNumber) AS num_employees
    FROM offices
    JOIN employees USING(officeCode)
    GROUP BY city
    ORDER BY num_employees DESC;
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

---

## Aliasing with AS

Aggregate column headers like `COUNT(employeeNumber)` are unreadable. Give them a name with `AS`. You can then reference that alias in `ORDER BY`.

You can also alias the `GROUP BY` column reference with a positional index (`GROUP BY 1` means "group by the first selected column") to reduce repetition.

In [ ]:
# Per-customer payment summary with readable column names
cur.execute("""
    SELECT customerName,
           COUNT(amount)  AS num_payments,
           MIN(amount)    AS min_payment,
           MAX(amount)    AS max_payment,
           AVG(amount)    AS avg_payment,
           SUM(amount)    AS total_spent
    FROM customers
    JOIN payments USING(customerNumber)
    GROUP BY 1
    ORDER BY total_spent DESC;
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
print(f"{len(df)} customers")
df.head()

---

## HAVING

`HAVING` filters on the *result of an aggregation*. It runs after `GROUP BY`, whereas `WHERE` runs before grouping (on individual rows). Use them together to first reduce the row set, then filter the grouped summaries.

```sql
-- Only cities with at least 5 customers
SELECT city, COUNT(customerNumber) AS n
FROM customers
GROUP BY 1
HAVING COUNT(customerNumber) >= 5;
```

In [ ]:
# Cities with 5 or more customers
cur.execute("""
    SELECT city, COUNT(customerNumber) AS num_customers
    FROM customers
    GROUP BY 1
    HAVING COUNT(customerNumber) >= 5;
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

---

## WHERE and HAVING Together

`WHERE` filters individual rows *before* grouping. `HAVING` filters groups *after* the aggregation. They are complementary, not redundant.

In [ ]:
# Customers who made at least 2 payments above $50,000
# WHERE filters out payments <= 50K first, then HAVING filters on the remaining count
cur.execute("""
    SELECT customerName,
           COUNT(amount) AS num_large_payments
    FROM customers
    JOIN payments USING(customerNumber)
    WHERE amount >= 50000
    GROUP BY customerName
    HAVING COUNT(amount) >= 2
    ORDER BY num_large_payments DESC;
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

---

## Practice: Babe Ruth Career Statistics

The `babe_ruth_stats` table holds Ruth's season-by-season hitting statistics (1914–1935).

Columns: `year`, `team`, `league`, `doubles`, `triples`, `hits`, `HR`, `games`, `runs`, `RBI`, `at_bats`, `BB`, `SB`, `SO`, `AVG`

In [ ]:
ruth_conn = sqlite3.connect('data/grouping_data_with_sql_lab/babe_ruth.db')
ruth = ruth_conn.cursor()

# Preview
ruth.execute("SELECT * FROM babe_ruth_stats LIMIT 3;")
df = pd.DataFrame(ruth.fetchall())
df.columns = [x[0] for x in ruth.description]
df

In [ ]:
# Total number of seasons Ruth played


In [ ]:
# Total seasons played for the NY Yankees specifically


In [ ]:
# The season with the most HR — return all columns for that row


In [ ]:
# The season with the fewest HR — return all columns


In [ ]:
# Career total HR


In [ ]:
# 5 worst HR seasons among seasons with at least 100 games played


In [ ]:
# Career batting average — use alias 'career_average'


In [ ]:
# Total years played (AS num_years) and total hits (AS total_hits) per team


In [ ]:
# Seasons where Ruth reached base (hits + BB) more than 300 times
# Compute the on_base column as hits + BB, alias it 'on_base'
# Return year and on_base only


---

## Summary

In this notebook you learned how to:

- Summarise a whole table with `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`
- Split summaries by category using `GROUP BY` (single or positional reference)
- Label aggregate columns cleanly with `AS`
- Filter on grouped results using `HAVING` (post-aggregation)
- Chain `WHERE` (pre-aggregation row filter) and `HAVING` (post-aggregation group filter) in the same query

Next: [04 — Joins](04_joins.ipynb)